# Tracking Quality Analysis

Compare benchmark results against optional human ground truth and rank each pipeline per scenario.

This notebook evaluates runtime, throughput, unique-count error, fragmentation proxies, entry/exit timing, and temporal interval overlap. The scripts emit one summary row per ID, which cannot measure IDF1, HOTA, MOTA, identity switches, or trajectory accuracy. Those metrics require per-frame predictions and per-frame ground-truth trajectories.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import linear_sum_assignment

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
EXPERIMENT_DIR = ROOT / "outputs" / "experiments"

# Set this to a specific aggregate CSV, or leave as None to use the newest one.
BENCHMARK_PATH = None
if BENCHMARK_PATH is None:
    candidates = sorted(EXPERIMENT_DIR.glob("scenario_benchmark_*.csv"), key=lambda path: path.stat().st_mtime)
    if not candidates:
        raise FileNotFoundError("Run scenario_benchmark.ipynb first or set BENCHMARK_PATH")
    benchmark_path = candidates[-1]
else:
    benchmark_path = Path(BENCHMARK_PATH).expanduser().resolve()

results = pd.read_csv(benchmark_path)
required_columns = {
    "scenario", "pipeline", "processing_seconds", "processing_fps",
    "unique_people", "zero_duration_tracks", "under_1s_tracks",
    "mean_track_duration", "telemetry_path",
}
missing = required_columns - set(results.columns)
if missing:
    raise ValueError(f"Benchmark CSV is missing columns: {sorted(missing)}")
print("Loaded:", benchmark_path)
results.head()

In [ ]:
results["zero_duration_ratio"] = (
    results["zero_duration_tracks"] / results["unique_people"].replace(0, np.nan)
)
results["under_1s_ratio"] = (
    results["under_1s_tracks"] / results["unique_people"].replace(0, np.nan)
)

summary = results.groupby(["scenario", "pipeline"], as_index=False).agg(
    processing_seconds_mean=("processing_seconds", "mean"),
    processing_seconds_std=("processing_seconds", "std"),
    processing_fps_mean=("processing_fps", "mean"),
    unique_people_mean=("unique_people", "mean"),
    unique_people_std=("unique_people", "std"),
    zero_duration_ratio_mean=("zero_duration_ratio", "mean"),
    under_1s_ratio_mean=("under_1s_ratio", "mean"),
    mean_track_duration=("mean_track_duration", "mean"),
)
summary

## Unique-person count ground truth

Fill `expected_unique_people` after manually labeling each scenario. Missing values are excluded from count-accuracy rankings.

In [ ]:
GROUND_TRUTH_COUNTS = pd.DataFrame({
    "scenario": sorted(results["scenario"].unique()),
    "expected_unique_people": pd.array(
        [pd.NA] * results["scenario"].nunique(), dtype="Float64"
    ),
})

# Example:
# GROUND_TRUTH_COUNTS.loc[
#     GROUND_TRUTH_COUNTS["scenario"] == "three_people_walking",
#     "expected_unique_people",
# ] = 3
GROUND_TRUTH_COUNTS

In [ ]:
count_truth = GROUND_TRUTH_COUNTS.dropna(subset=["expected_unique_people"])
count_evaluation = summary.merge(count_truth, on="scenario", how="inner")
if count_evaluation.empty:
    print("Enter expected counts above to calculate count errors.")
else:
    count_evaluation["count_absolute_error"] = (
        count_evaluation["unique_people_mean"] - count_evaluation["expected_unique_people"]
    ).abs()
    count_evaluation["count_relative_error"] = (
        count_evaluation["count_absolute_error"]
        / count_evaluation["expected_unique_people"].replace(0, np.nan)
    )
    count_evaluation["count_rank"] = count_evaluation.groupby("scenario")["count_absolute_error"].rank(method="min")
count_evaluation

## Optional entry/exit interval ground truth

Provide a CSV with `scenario,person_id,entry_seconds,exit_seconds`. Prediction IDs and ground-truth IDs do not need to match; intervals are paired with Hungarian assignment using temporal IoU.

In [ ]:
GROUND_TRUTH_INTERVALS_PATH = None

def interval_iou(entry_a, exit_a, entry_b, exit_b):
    intersection = max(0.0, min(exit_a, exit_b) - max(entry_a, entry_b))
    union = max(exit_a, exit_b) - min(entry_a, entry_b)
    return intersection / union if union > 0 else float(entry_a == entry_b and exit_a == exit_b)

def evaluate_intervals(predicted, truth):
    if predicted.empty or truth.empty:
        return {
            "matched_intervals": 0,
            "mean_interval_iou": np.nan,
            "entry_mae": np.nan,
            "exit_mae": np.nan,
            "unmatched_predictions": len(predicted),
            "unmatched_ground_truth": len(truth),
            "precision": 0.0 if len(predicted) else np.nan,
            "recall": 0.0 if len(truth) else np.nan,
        }

    ious = np.array([
        [
            interval_iou(p.entry_seconds, p.exit_seconds, g.entry_seconds, g.exit_seconds)
            for g in truth.itertuples()
        ]
        for p in predicted.itertuples()
    ])
    rows, columns = linear_sum_assignment(1.0 - ious)
    valid = [(row, column) for row, column in zip(rows, columns) if ious[row, column] > 0]
    entry_errors = [
        abs(predicted.iloc[row]["entry_seconds"] - truth.iloc[column]["entry_seconds"])
        for row, column in valid
    ]
    exit_errors = [
        abs(predicted.iloc[row]["exit_seconds"] - truth.iloc[column]["exit_seconds"])
        for row, column in valid
    ]
    matched = len(valid)
    return {
        "matched_intervals": matched,
        "mean_interval_iou": np.mean([ious[row, column] for row, column in valid]) if valid else np.nan,
        "entry_mae": np.mean(entry_errors) if entry_errors else np.nan,
        "exit_mae": np.mean(exit_errors) if exit_errors else np.nan,
        "unmatched_predictions": len(predicted) - matched,
        "unmatched_ground_truth": len(truth) - matched,
        "precision": matched / len(predicted),
        "recall": matched / len(truth),
    }

interval_records = []
if GROUND_TRUTH_INTERVALS_PATH is not None:
    interval_truth = pd.read_csv(GROUND_TRUTH_INTERVALS_PATH)
    interval_required = {"scenario", "person_id", "entry_seconds", "exit_seconds"}
    missing = interval_required - set(interval_truth.columns)
    if missing:
        raise ValueError(f"Interval ground truth is missing columns: {sorted(missing)}")

    for run in results.itertuples():
        predicted = pd.read_csv(run.telemetry_path)
        truth = interval_truth[interval_truth["scenario"] == run.scenario].reset_index(drop=True)
        metrics = evaluate_intervals(predicted, truth)
        interval_records.append({"scenario": run.scenario, "pipeline": run.pipeline, "repetition": run.repetition, **metrics})

interval_evaluation = pd.DataFrame(interval_records)
interval_evaluation

## Weighted recommendation

Adjust weights to reflect deployment priorities. Lower count error, fragmentation, temporal error, and runtime are better. Scores are normalized within each scenario, so they compare pipelines only on the scenarios provided.

In [ ]:
WEIGHTS = {
    "count_error": 0.45,
    "fragmentation": 0.20,
    "temporal_error": 0.20,
    "runtime": 0.15,
}

def normalize_within_scenario(frame, column):
    def normalize(series):
        spread = series.max() - series.min()
        return (series - series.min()) / spread if spread > 0 else pd.Series(0.0, index=series.index)
    return frame.groupby("scenario")[column].transform(normalize)

recommendations = summary.copy()
recommendations = recommendations.merge(
    count_evaluation[["scenario", "pipeline", "count_absolute_error"]]
    if not count_evaluation.empty
    else pd.DataFrame(columns=["scenario", "pipeline", "count_absolute_error"]),
    on=["scenario", "pipeline"],
    how="left",
)
recommendations["count_component"] = normalize_within_scenario(
    recommendations, "count_absolute_error"
) if recommendations["count_absolute_error"].notna().all() else 0.0
recommendations["fragmentation_component"] = normalize_within_scenario(
    recommendations, "under_1s_ratio_mean"
)
recommendations["runtime_component"] = normalize_within_scenario(
    recommendations, "processing_seconds_mean"
)
recommendations["temporal_component"] = 0.0

if not interval_evaluation.empty:
    temporal = interval_evaluation.groupby(["scenario", "pipeline"], as_index=False).agg(
        temporal_error=("mean_interval_iou", lambda values: 1.0 - values.mean())
    )
    recommendations = recommendations.merge(temporal, on=["scenario", "pipeline"], how="left")
    recommendations["temporal_component"] = normalize_within_scenario(
        recommendations, "temporal_error"
    ).fillna(0.0)

recommendations["weighted_score"] = (
    WEIGHTS["count_error"] * recommendations["count_component"]
    + WEIGHTS["fragmentation"] * recommendations["fragmentation_component"]
    + WEIGHTS["temporal_error"] * recommendations["temporal_component"]
    + WEIGHTS["runtime"] * recommendations["runtime_component"]
)
recommendations["scenario_rank"] = recommendations.groupby("scenario")["weighted_score"].rank(method="min")
recommendations.sort_values(["scenario", "scenario_rank"])[
    ["scenario", "pipeline", "weighted_score", "scenario_rank"]
]

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
for axis, metric, title in [
    (axes[0], "processing_fps_mean", "Throughput (higher is better)"),
    (axes[1], "under_1s_ratio_mean", "Short-track ratio (lower is better)"),
    (axes[2], "weighted_score", "Weighted score (lower is better)"),
]:
    recommendations.pivot(index="scenario", columns="pipeline", values=metric).plot(kind="bar", ax=axis, title=title)
    axis.tick_params(axis="x", rotation=30)
fig.tight_layout()
plt.show()

run_stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
summary_path = EXPERIMENT_DIR / f"tracking_quality_summary_{run_stamp}.csv"
recommendations.to_csv(summary_path, index=False)
print("Saved:", summary_path)